To Extract Dataset from ZIP File

In [0]:
import zipfile
import os

zip_path = "/Volumes/workspace/default/ml_assignment/csv_pus.zip"
extract_path = "/Volumes/workspace/default/ml_assignment/"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Unzip completed")

Loading Dataset into Spark DataFrame

In [0]:
data_path = "/Volumes/workspace/default/ml_assignment/psam_pus*.csv"

df = (spark.read
      .option("header", "true")
      .option("inferSchema", "true")
      .csv(data_path))

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.show(5, truncate=False)

Inspecting Dataset Structure (Columns and Schema)

In [0]:
# Show column names
print(df.columns)

# Show schema (data types)
df.printSchema()

Creating Target Variable (Income Classification Label)

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "income_label",
    when(col("PINCP") >= 50000, 1).otherwise(0)
)

df.select("PINCP", "income_label").show(10)

Checking Class Distribution of Target Variable

In [0]:
df.groupBy("income_label").count().show()


Selecting Relevant Features for Analysis

In [0]:
selected_cols = [
    "AGEP", "SEX", "SCHL", "MAR", "OCCP",
    "WKHP", "COW", "CIT", "RAC1P",
    "ENG", "DIS", "HICOV", "NATIVITY",
    "income_label"
]

df_selected = df.select(selected_cols)

df_selected.show(5)

Checking Missing Values in Selected Features

In [0]:
from pyspark.sql.functions import col, sum

# To count missing values per column
missing_counts = df_selected.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_selected.columns
])

missing_counts.show()

Removing Column with High Missing Values

In [0]:
df_selected = df_selected.drop("ESP", "ENG")

df_selected.show(5)

Handling Missing Values Using Median Imputation

In [0]:
from pyspark.ml.feature import Imputer

columns_to_impute = ["SCHL", "MAR", "OCCP", "WKHP", "COW"]

imputer = Imputer(
    inputCols=columns_to_impute,
    outputCols=columns_to_impute
).setStrategy("median")

df_selected = imputer.fit(df_selected).transform(df_selected)

df_selected.show(5)

Verifying Missing Values After Preprocessing

In [0]:
from pyspark.sql.functions import sum, col

check_cols = ["AGEP","SEX","SCHL","MAR","OCCP","WKHP","COW","CIT","RAC1P","income_label"]
df_selected.select([sum(col(c).isNull().cast("int")).alias(c) for c in check_cols]).show()

Creating Feature Vector for Machine Learning Models

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "AGEP", "SEX", "SCHL", "MAR", "OCCP",
    "WKHP", "COW", "CIT", "RAC1P",
    "DIS", "HICOV", "NATIVITY"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

df_final = assembler.transform(df_selected)

df_final.select("features", "income_label").show(5, truncate=False)

Preview Final Features and Target Variable

In [0]:
df_final.select("features","income_label").show(3, truncate=False)

Filtering Invalid Age Values

In [0]:
df_final = df_final.filter(df_final.AGEP > 0)

Checking Age Range After Filtering

In [0]:
from pyspark.sql.functions import min, max

df_final.select(min("AGEP").alias("min_age"), max("AGEP").alias("max_age")).show()

Final Verification of Dataset Structure and Size after Filtering

In [0]:
print("Columns:", df_final.columns)
print("Total rows:", df_final.count())

Checking Final Class Distribution After Preprocessing

In [0]:
df_final.groupBy("income_label").count().show()

Spliting Dataset into Training and Testing Sets

In [0]:
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

Verifying Training and Testing Data Distribution

In [0]:
print("Train rows:", train_data.count())
print("Test rows:", test_data.count())

train_data.groupBy("income_label").count().show()
test_data.groupBy("income_label").count().show()

Preview Training and Testing Data

In [0]:
train_data.show(5)
test_data.show(5)

Training Logistic Regression Model and Generate Predictions

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="income_label"
)

lr_model = lr.fit(train_data)

predictions_lr = lr_model.transform(test_data)

predictions_lr.select(
    "income_label",
    "prediction",
    "probability"
).show(5, truncate=False)

Evaluating Logistic Regression Model Performance

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC
lr_evaluator_auc = BinaryClassificationEvaluator(
    labelCol="income_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
lr_auc = lr_evaluator_auc.evaluate(predictions_lr)

# Accuracy
lr_evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="accuracy"
)
lr_acc = lr_evaluator_acc.evaluate(predictions_lr)

# Precision
lr_evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)
lr_precision = lr_evaluator_precision.evaluate(predictions_lr)

# Recall
lr_evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="weightedRecall"
)
lr_recall = lr_evaluator_recall.evaluate(predictions_lr)

# F1
lr_evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="f1"
)
lr_f1 = lr_evaluator_f1.evaluate(predictions_lr)

print("Logistic Regression AUC:", lr_auc)
print("Logistic Regression Accuracy:", lr_acc)
print("Logistic Regression Precision:", lr_precision)
print("Logistic Regression Recall:", lr_recall)
print("Logistic Regression F1:", lr_f1)

Preview Logistic Regression Predictions

In [0]:
predictions_lr.select("income_label", "prediction").show(10)

Generating Formatted Confusion Matrix Table — Logistic Regression

In [0]:
lr_confusion_matrix_df = predictions_lr.groupBy("income_label") \
    .pivot("prediction") \
    .count() \
    .fillna(0)

lr_confusion_matrix_df.show()

Training Decision Tree Model and Generate Predictions

In [0]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    labelCol="income_label",
    featuresCol="features",
    maxDepth=10,
    minInstancesPerNode=50
)

dt_model = dt.fit(train_data)
predictions_dt = dt_model.transform(test_data)

predictions_dt.select("income_label", "prediction", "probability").show(5, truncate=False)

Evaluating Decision Tree Model Performance

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC
dt_auc_eval = BinaryClassificationEvaluator(
    labelCol="income_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
dt_auc = dt_auc_eval.evaluate(predictions_dt)

# Accuracy
dt_acc_eval = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="accuracy"
)
dt_acc = dt_acc_eval.evaluate(predictions_dt)

# Precision
dt_precision_eval = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)
dt_precision = dt_precision_eval.evaluate(predictions_dt)

# Recall
dt_recall_eval = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="weightedRecall"
)
dt_recall = dt_recall_eval.evaluate(predictions_dt)

# F1
dt_f1_eval = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="f1"
)
dt_f1 = dt_f1_eval.evaluate(predictions_dt)

print("Decision Tree AUC:", dt_auc)
print("Decision Tree Accuracy:", dt_acc)
print("Decision Tree Precision:", dt_precision)
print("Decision Tree Recall:", dt_recall)
print("Decision Tree F1:", dt_f1)

Generating Formatted Confusion Matrix Table — Decision Tree

In [0]:
dt_confusion_matrix_df = (predictions_dt.groupBy("income_label")
                          .pivot("prediction")
                          .count()
                          .fillna(0))

dt_confusion_matrix_df.show()

Training Random Forest Model and Generate Predictions

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="income_label",
    featuresCol="features",
    numTrees=50,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_data)
predictions_rf = rf_model.transform(test_data)
predictions_rf.select("income_label", "prediction", "probability").show(5, truncate=False)

Evaluating Random Forest Model Performance

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC
rf_auc = BinaryClassificationEvaluator(
    labelCol="income_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions_rf)

# Accuracy
rf_acc = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions_rf)

# Precision
rf_precision = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions_rf)

# Recall
rf_recall = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions_rf)

# F1
rf_f1 = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions_rf)

print("Random Forest AUC:", rf_auc)
print("Random Forest Accuracy:", rf_acc)
print("Random Forest Precision:", rf_precision)
print("Random Forest Recall:", rf_recall)
print("Random Forest F1:", rf_f1)

Generating Formatted Confusion Matrix Table — Random Forest

In [0]:
rf_confusion_matrix_df = (predictions_rf.groupBy("income_label")
                          .pivot("prediction")
                          .count()
                          .fillna(0))
rf_confusion_matrix_df.show()

Extracting feature importance from Random Forest

In [0]:
import pandas as pd

feature_names = [
    "AGEP", "SEX", "SCHL", "MAR", "OCCP",
    "WKHP", "COW", "CIT", "RAC1P",
    "DIS", "HICOV", "NATIVITY"
]

importance = rf_model.featureImportances.toArray()

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

feature_importance_df

Visualize Feature Importance Using Bar Chart

In [0]:
import matplotlib.pyplot as plt

feature_importance_df.plot(kind="bar", x="Feature", y="Importance", legend=False)
plt.title("Random Forest Feature Importance")
plt.ylabel("Importance Score")
plt.show()

Training Gradient Boosted Tree Model and Generate Predictions

In [0]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(
    labelCol="income_label",
    featuresCol="features",
    maxIter=50,
    maxDepth=5,
    seed=42
)

gbt_model = gbt.fit(train_data)
predictions_gbt = gbt_model.transform(test_data)
predictions_gbt.select("income_label", "prediction", "probability").show(5, truncate=False)

Evaluating Gradient Boosted Tree Model Performance

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC
gbt_auc = BinaryClassificationEvaluator(
    labelCol="income_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions_gbt)

# Accuracy
gbt_acc = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions_gbt)

# Precision
gbt_precision = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(predictions_gbt)

# Recall
gbt_recall = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(predictions_gbt)

# F1
gbt_f1 = MulticlassClassificationEvaluator(
    labelCol="income_label",
    predictionCol="prediction",
    metricName="f1"
).evaluate(predictions_gbt)

print("GBT AUC:", gbt_auc)
print("GBT Accuracy:", gbt_acc)
print("GBT Precision:", gbt_precision)
print("GBT Recall:", gbt_recall)
print("GBT F1:", gbt_f1)

Generating Formatted Confusion Matrix Table — Gradient Boosted Trees

In [0]:
gbt_confusion_matrix_df = (predictions_gbt.groupBy("income_label")
                           .pivot("prediction")
                           .count()
                           .fillna(0))

gbt_confusion_matrix_df.show()

Final Baseline model comparison table

In [0]:
results = [
    ("Logistic Regression", lr_auc, lr_acc, lr_precision, lr_recall, lr_f1),
    ("Decision Tree",       dt_auc, dt_acc, dt_precision, dt_recall, dt_f1),
    ("Random Forest",       rf_auc, rf_acc, rf_precision, rf_recall, rf_f1),
    ("Gradient Boosted Trees", gbt_auc, gbt_acc, gbt_precision, gbt_recall, gbt_f1)
]

results_df = spark.createDataFrame(
    results,
    ["Model", "AUC", "Accuracy", "Precision", "Recall", "F1"]
)

results_df.orderBy(results_df["F1"].desc()).show(truncate=False)

Creating and Exporting Final Combined Model Metrics Table

In [0]:
# Create and Export Final Combined Model Metrics Table

out_path = "/Volumes/workspace/default/ml_assignment/model_metrics_final_single"

model_metrics_final = spark.createDataFrame([
    ("Logistic Regression", lr_auc, lr_acc, lr_precision, lr_recall, lr_f1),
    ("Decision Tree", dt_auc, dt_acc, dt_precision, dt_recall, dt_f1),
    ("Random Forest", rf_auc, rf_acc, rf_precision, rf_recall, rf_f1),
    ("Gradient Boosted Trees", gbt_auc, gbt_acc, gbt_precision, gbt_recall, gbt_f1)
], ["Model", "AUC", "Accuracy", "Precision", "Recall", "F1"])

# Verification
print("Rows:", model_metrics_final.count())
model_metrics_final.orderBy("Model").show(truncate=False)

# Output file for Tableau
(model_metrics_final
 .coalesce(1)
 .write.mode("overwrite")
 .option("header", True)
 .csv(out_path))

print(f"✅ Exported combined metrics to: {out_path}")

Spark ML Configuration: Set Temporary DFS Path for CrossValidator

In [0]:
import os

# Required for Databricks Serverless ML caching
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/ml_assignment/sparkml_tmp"

print("SPARKML_TEMP_DFS_PATH set to:", os.environ["SPARKML_TEMP_DFS_PATH"])

Creating Spark ML Temporary Storage Folder

In [0]:
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/ml_assignment/sparkml_tmp")

print("Temporary folder created successfully")

CrossValidator Prep: Train / Validation / Test Split 

In [0]:
# Stratified split (keeps class balance similar across splits)
fractions = {0: 0.8, 1: 0.8}   # 80% for train_val
train_val = df_final.sampleBy("income_label", fractions=fractions, seed=42)

test_data = df_final.subtract(train_val)

# Split train_val into train and validation
fractions2 = {0: 0.75, 1: 0.75}   # 75% of train_val -> train
train_data = train_val.sampleBy("income_label", fractions=fractions2, seed=43)

val_data = train_val.subtract(train_data)

print("Train:", train_data.count())
print("Val:", val_data.count())
print("Test:", test_data.count())

print("Train distribution:")
train_data.groupBy("income_label").count().show()

print("Validation distribution:")
val_data.groupBy("income_label").count().show()

print("Test distribution:")
test_data.groupBy("income_label").count().show()

CrossValidator: Tune Random Forest (MLlib) and pick best

In [0]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Parameter grid
paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [20, 50])
             .addGrid(rf.maxDepth, [5, 10])
             .addGrid(rf.minInstancesPerNode, [20, 50])
             .build())

# Evaluator
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="income_label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# CrossValidator
cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_auc,
    numFolds=3,
    parallelism=2,  
    seed=42
)

# Train CV model
cv_rf_model = cv.fit(train_data)

# Best model
best_rf_model = cv_rf_model.bestModel

print("Best RF Parameters:")
print("numTrees:", best_rf_model.getNumTrees)
print("maxDepth:", best_rf_model.getOrDefault("maxDepth"))
print("minInstancesPerNode:", best_rf_model.getOrDefault("minInstancesPerNode"))

Evaluating Tuned Random Forest on Validation and Test

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

def eval_5_metrics(pred_df, label_col="income_label"):
    auc_eval = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderROC")
    acc_eval = MulticlassClassificationEvaluator(labelCol=label_col, metricName="accuracy")
    prec_eval = MulticlassClassificationEvaluator(labelCol=label_col, metricName="weightedPrecision")
    rec_eval  = MulticlassClassificationEvaluator(labelCol=label_col, metricName="weightedRecall")
    f1_eval   = MulticlassClassificationEvaluator(labelCol=label_col, metricName="f1")
    
    return {
        "AUC": auc_eval.evaluate(pred_df),
        "Accuracy": acc_eval.evaluate(pred_df),
        "Precision": prec_eval.evaluate(pred_df),
        "Recall": rec_eval.evaluate(pred_df),
        "F1": f1_eval.evaluate(pred_df)
    }

# Validation Evaluation
pred_rf_val = best_rf_model.transform(val_data)
rf_val_metrics = eval_5_metrics(pred_rf_val)

print("Tuned RF (VAL):", rf_val_metrics)

# Test Evaluation
pred_rf_test = best_rf_model.transform(test_data)
rf_test_metrics = eval_5_metrics(pred_rf_test)

print("Tuned RF (TEST):", rf_test_metrics)

# Storing metrics for final comparison table
tuned_rf_auc = rf_test_metrics["AUC"]
tuned_rf_acc = rf_test_metrics["Accuracy"]
tuned_rf_precision = rf_test_metrics["Precision"]
tuned_rf_recall = rf_test_metrics["Recall"]
tuned_rf_f1 = rf_test_metrics["F1"]

# Confusion Matrix
cm_rf_test = pred_rf_test.groupBy("income_label").pivot("prediction").count().fillna(0)
cm_rf_test.show()

Model Serialization: save best tuned RF model

In [0]:
model_path = "/Volumes/workspace/default/ml_assignment/best_rf_model"

best_rf_model.write().overwrite().save(model_path)

print("Model saved at:", model_path)

Final Model Comparison Including Tuned Random Forest

In [0]:
final_results = [
    ("Logistic Regression", lr_auc, lr_acc, lr_precision, lr_recall, lr_f1),
    ("Decision Tree", dt_auc, dt_acc, dt_precision, dt_recall, dt_f1),
    ("Random Forest", rf_auc, rf_acc, rf_precision, rf_recall, rf_f1),
    ("Gradient Boosted Trees", gbt_auc, gbt_acc, gbt_precision, gbt_recall, gbt_f1),
    ("Tuned Random Forest", tuned_rf_auc, tuned_rf_acc, tuned_rf_precision, tuned_rf_recall, tuned_rf_f1)
]

final_results_df = spark.createDataFrame(
    final_results,
    ["Model", "AUC", "Accuracy", "Precision", "Recall", "F1"]
)

final_results_df.orderBy(final_results_df["F1"].desc()).show(truncate=False)

 scikit-learn Baseline (single node) — Logistic Regression on a small sample

In [0]:
# scikit-learn Baseline (single node) — Logistic Regression on a small sample

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

# 1) Pulling a small sample to the driver
pdf = (
    df_final
    .select("income_label", "features")
    .sample(fraction=0.002, seed=42)  
    .limit(200000)                    
    .toPandas()
)

# 2) Converting Spark ML vector -> numpy matrix
X = np.vstack(pdf["features"].apply(lambda v: v.toArray()).values)
y = pdf["income_label"].astype(int).values

# 3) Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4) Scale + Logistic Regression
model = Pipeline(steps=[
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("lr", LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

# 5) Predict probabilities + labels
proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

# 6) Metrics (5)
print("sklearn LR AUC:", roc_auc_score(y_test, proba))
print("sklearn LR Accuracy:", accuracy_score(y_test, pred))
print("sklearn LR Precision:", precision_score(y_test, pred, zero_division=0))
print("sklearn LR Recall:", recall_score(y_test, pred, zero_division=0))
print("sklearn LR F1:", f1_score(y_test, pred, zero_division=0))

Extracting Feature Importance from Gradient Boosted Trees

In [0]:
import pandas as pd

feature_names = assembler.getInputCols()
gbt_importance = gbt_model.featureImportances.toArray()

gbt_feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": gbt_importance
}).sort_values(by="Importance", ascending=False)

print(gbt_feature_importance_df.shape)
display(gbt_feature_importance_df)

In [0]:
spark.createDataFrame(gbt_feature_importance_df) \
  .coalesce(1) \
  .write.mode("overwrite") \
  .option("header", True) \
  .csv("/Volumes/workspace/default/ml_assignment/feature_importance/gbt_feature_importance")

Visualize Feature Importance of Gradient Boosted Trees

In [0]:
import matplotlib.pyplot as plt

gbt_feature_importance_df.plot(
    kind="bar",
    x="Feature",
    y="Importance",
    legend=False
)

plt.title("Gradient Boosted Trees Feature Importance")
plt.ylabel("Importance Score")
plt.show()

Export Final Model Comparison Table for Tableau

In [0]:
final_results_df.write.mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/ml_assignment/model_metrics_final")

print("Final model metrics exported: /Volumes/workspace/default/ml_assignment/model_metrics_final")

Export Feature Importance for Tableau

In [0]:
# Export Feature Importance for Tableau (RF)
feature_names = assembler.getInputCols()
rf_importance = rf_model.featureImportances.toArray()

rf_feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_importance
}).sort_values(by="Importance", ascending=False)

display(rf_feature_importance_df)

spark.createDataFrame(rf_feature_importance_df) \
  .coalesce(1) \
  .write.mode("overwrite") \
  .option("header", True) \
  .csv("/Volumes/workspace/default/ml_assignment/feature_importance/rf_feature_importance_single")

In [0]:
# Export Feature Importance for Tableau (Random Forest - Baseline)

import pandas as pd

rf_importance = rf_model.featureImportances.toArray()
rf_feature_importance_df = pd.DataFrame({
    "Feature": assembler.getInputCols(),
    "Importance": rf_importance
}).sort_values(by="Importance", ascending=False)

spark.createDataFrame(rf_feature_importance_df).write.mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/ml_assignment/feature_importance/rf_feature_importance")

print("RF feature importance exported: /Volumes/workspace/default/ml_assignment/feature_importance/rf_feature_importance")

Export Data Quality Summary for Tableau

In [0]:
# Export Data Quality Summary for Tableau (Missing Values + Class Distribution)

from pyspark.sql.functions import sum as Fsum, col

# Missing values table (one row with missing counts per column)
missing_summary = df_selected.select([
    Fsum(col(c).isNull().cast("int")).alias(c) for c in df_selected.columns
])

# Class distribution table
class_distribution = df_selected.groupBy("income_label").count()

missing_summary.write.mode("overwrite").option("header", True) \
    .csv("/Volumes/workspace/default/ml_assignment/tableau_data_quality/missing_summary")

class_distribution.write.mode("overwrite").option("header", True) \
    .csv("/Volumes/workspace/default/ml_assignment/tableau_data_quality/class_distribution")

print("Data quality exports completed")

Export Processed Dataset Sample for Visualization

In [0]:
# Export Processed Dataset Sample for Tableau 

sample_df = df_final.sample(fraction=0.1, seed=42).drop("features")

sample_df.write.mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/ml_assignment/income_sample")

print("Dataset sample exported: /Volumes/workspace/default/ml_assignment/income_sample")

Export Baseline Model Metrics

In [0]:
results_df.write.mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/ml_assignment/model_metrics_baseline")

print("Baseline model metrics exported successfully")

Export Model Predictions for Visualization

In [0]:
# Export Model Predictions for Tableau (baseline + tuned, all features)

from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col

base_path = "/Volumes/workspace/default/ml_assignment/model_exports"
feature_cols = assembler.getInputCols()

def export_predictions(pred_df, model_name):
    out = (pred_df
           .withColumn("prob_array", vector_to_array(col("probability")))
           .withColumn("prob_class1", col("prob_array")[1])
           .select(
               *feature_cols,
               "income_label", "prediction", "prob_class1"
           ))

    (out.write
        .mode("overwrite")
        .option("header", True)
        .csv(f"{base_path}/{model_name}_predictions"))

    print(f"{model_name} predictions exported -> {base_path}/{model_name}_predictions")

# Baseline models
export_predictions(predictions_lr,  "lr")
export_predictions(predictions_dt,  "dt")
export_predictions(predictions_rf,  "rf")
export_predictions(predictions_gbt, "gbt")

# Tuned best model
tuned_rf_test_pred = best_rf_model.transform(test_data)
export_predictions(tuned_rf_test_pred, "rf_tuned")

Export Scalability Timing Results for Tableau (Runtime vs Data Size)

In [0]:
# Export Scalability Timing Results for Tableau (Runtime vs Data Size)

import time
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

sizes = [0.1, 0.5, 1.0]
timing_rows = []

auc_eval = BinaryClassificationEvaluator(labelCol="income_label", metricName="areaUnderROC")

for frac in sizes:
    df_tmp = df_final.sample(fraction=frac, seed=42)
    train_tmp, test_tmp = df_tmp.randomSplit([0.8, 0.2], seed=42)

    gbt_tmp = GBTClassifier(
        labelCol="income_label",
        featuresCol="features",
        maxIter=30,
        maxDepth=5,
        seed=42
    )

    start = time.time()
    model_tmp = gbt_tmp.fit(train_tmp)
    preds_tmp = model_tmp.transform(test_tmp)
    auc_tmp = auc_eval.evaluate(preds_tmp)
    elapsed = time.time() - start

    timing_rows.append((frac, train_tmp.count(), elapsed, auc_tmp))

timing_df = spark.createDataFrame(
    timing_rows,
    ["data_fraction", "train_rows", "train_eval_seconds", "auc"]
)

timing_df.write.mode("overwrite").option("header", True) \
    .csv("/Volumes/workspace/default/ml_assignment/scalability/timing_results")

timing_df.show()
print("Scalability timing exported successfully")

In [0]:
# Export Scalability Timing Results for Tableau (Runtime vs Data Size)

import time
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# More dataset sizes for stronger scalability analysis
sizes = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]

timing_rows = []

auc_eval = BinaryClassificationEvaluator(
    labelCol="income_label",
    metricName="areaUnderROC"
)

for frac in sizes:

    # Sample dataset
    df_tmp = df_final.sample(fraction=frac, seed=42)

    # Train/Test split
    train_tmp, test_tmp = df_tmp.randomSplit([0.8, 0.2], seed=42)

    # Model configuration
    gbt_tmp = GBTClassifier(
        labelCol="income_label",
        featuresCol="features",
        maxIter=30,
        maxDepth=5,
        seed=42
    )

    # Measure training + evaluation time
    start = time.time()

    model_tmp = gbt_tmp.fit(train_tmp)
    preds_tmp = model_tmp.transform(test_tmp)

    auc_tmp = auc_eval.evaluate(preds_tmp)

    elapsed = time.time() - start

    # Store results
    timing_rows.append((
        frac,
        train_tmp.count(),
        elapsed,
        auc_tmp
    ))

# Create Spark DataFrame
timing_df = spark.createDataFrame(
    timing_rows,
    ["data_fraction", "train_rows", "train_eval_seconds", "auc"]
)

# Export to single CSV file for Tableau
timing_df.coalesce(1).write.mode("overwrite") \
.option("header", True) \
.csv("/Volumes/workspace/default/ml_assignment/scalability/scalability_timing")

# Display results
timing_df.show()

print("Scalability timing exported successfully")

Check Availability of Variables After Session Reconnect

In [0]:
print("df exists?", "df" in globals())
print("df_selected exists?", "df_selected" in globals())
print("df_final exists?", "df_final" in globals())